# 🧠 MRI → PET Synthesis for Alzheimer's Disease

**Publication-quality, production-ready pipeline**  
Optimized for Google Colab T4/A100 GPU

---

## Pipeline Overview

```
Dataset Inspection  →  Preprocessing  →  Leakage Verification
     ↓
Model Training  →  Evaluation  →  Visualization  →  Inference
```

**Supported datasets:** ADNI · AIBL · OASIS-3 · Custom  
**Supported tracers:** FDG · Florbetapir · Florbetaben · Flortaucipir  
**Default model:** 3D Residual Attention U-Net

> ⚠️ **Before running:** Make sure your dataset is placed under  
> `/content/drive/MyDrive/Google Colabs/data/`

---
## 📦 Section 0 — Install Dependencies

In [ ]:
# ── Install all required packages ──────────────────────────────────────────
import subprocess, sys

PACKAGES = [
    "torch==2.1.0",
    "torchvision==0.16.0",
    "monai==1.3.0",
    "torchio==0.19.6",
    "nibabel==5.1.0",
    "SimpleITK==2.3.1",
    "numpy==1.26.4",
    "scipy==1.12.0",
    "scikit-image==0.22.0",
    "pandas==2.2.0",
    "matplotlib==3.8.3",
    "seaborn==0.13.2",
    "pyyaml==6.0.1",
    "pydantic==2.6.1",
    "tqdm==4.66.2",
    "rich==13.7.0",
]

print("Installing packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + PACKAGES,
    check=True
)
print("✅ All packages installed.")

In [ ]:
# ── Clone / mount the pipeline source code ─────────────────────────────────
import os
from pathlib import Path

# If the pipeline is on Drive, adjust this path.
# Otherwise clone from GitHub:
# !git clone https://github.com/your-org/alzheimer-mri-pet.git /content/alzheimer-mri-pet

PIPELINE_ROOT = Path("/content/alzheimer-mri-pet")
SRC_PATH = PIPELINE_ROOT / "src"

# Add src/ to Python path
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Pipeline root : {PIPELINE_ROOT}")
print(f"Source path   : {SRC_PATH}")
print(f"✅ Source path added to sys.path.")

---
## ⚙️ Section 1 — Environment Setup

In [ ]:
# ── Mount Google Drive ──────────────────────────────────────────────────────
from engine.colab import (
    mount_google_drive,
    check_gpu_available,
    check_disk_space,
    report_hardware,
    set_reproducibility,
)

# Mount Drive
drive_mounted = mount_google_drive("/content/drive")

# Abort if no GPU
check_gpu_available()

# Report hardware
hw_info = report_hardware()

# Check local disk space
check_disk_space(min_gb=10)

In [ ]:
# ── Load configuration ──────────────────────────────────────────────────────
from config import load_config, config_to_dict

DEFAULT_CONFIG  = PIPELINE_ROOT / "config" / "default.yaml"

# Optional: point to an experiment override
# EXPERIMENT_CONFIG = PIPELINE_ROOT / "config" / "experiments" / "my_run.yaml"
EXPERIMENT_CONFIG = None

cfg = load_config(DEFAULT_CONFIG, EXPERIMENT_CONFIG)

print("\n📋 Active configuration:")
print(f"  Dataset        : {cfg.data.dataset}")
print(f"  PET tracer     : {cfg.data.pet_tracer}")
print(f"  Model          : {cfg.model.name}")
print(f"  Loss           : L1×{cfg.loss.l1_weight} + SSIM×{cfg.loss.ssim_weight}")
print(f"  Max epochs     : {cfg.training.max_epochs}")
print(f"  Batch size     : {cfg.training.batch_size}")
print(f"  Patch size     : {cfg.patch.size}")
print(f"  AMP            : {cfg.training.amp_enabled}")
print(f"  Grad accum     : {cfg.training.gradient_accumulation_steps}")
print("\n✅ Config loaded and validated.")

In [ ]:
# ── Set reproducibility ─────────────────────────────────────────────────────
set_reproducibility(
    seed=cfg.reproducibility.random_seed,
    deterministic=cfg.reproducibility.deterministic,
    cudnn_benchmark=cfg.reproducibility.cudnn_benchmark,
)
print(f"✅ Seed = {cfg.reproducibility.random_seed}, deterministic = {cfg.reproducibility.deterministic}")

---
## 🔍 Section 2 — Dataset Inspection

> **Never assume dataset structure.**  
> The inspection report must be reviewed before any preprocessing begins.

In [ ]:
# ── Select dataset adapter ──────────────────────────────────────────────────
from pathlib import Path
from data.adapters.adni   import ADNIAdapter
from data.adapters.aibl   import AIBLAdapter
from data.adapters.oasis  import OASISAdapter
from data.adapters.custom import CustomAdapter

DATA_ROOT = Path(cfg.environment.data_root)

# ── Change this to match your dataset ──────────────────────────────────────
DATASET = cfg.data.dataset   # 'ADNI' | 'AIBL' | 'OASIS' | 'custom'

ADAPTER_MAP = {
    "ADNI"   : ADNIAdapter(mri_modality=cfg.data.mri_modality, pet_tracer=cfg.data.pet_tracer),
    "AIBL"   : AIBLAdapter(),
    "OASIS"  : OASISAdapter(),
    "custom" : CustomAdapter(),
}

if DATASET not in ADAPTER_MAP:
    raise ValueError(f"Unknown dataset: '{DATASET}'. Choose from {list(ADAPTER_MAP.keys())}")

adapter = ADAPTER_MAP[DATASET]
print(f"✅ Using adapter: {adapter.dataset_name}")

In [ ]:
# ── Discover subjects ───────────────────────────────────────────────────────
records = adapter.discover_subjects(DATA_ROOT)
print(f"\n✅ Discovered {len(records)} valid MRI/PET pairs.")

In [ ]:
# ── Generate Dataset Summary Report ────────────────────────────────────────
# ⚠️  READ THIS REPORT CAREFULLY before proceeding.

from data.inspector import generate_inspection_report

REPORT_PATH = Path(cfg.environment.output_root) / "inspection_report.txt"

report = generate_inspection_report(
    records=records,
    dataset_name=adapter.dataset_name,
    sample_size=5,
    output_path=REPORT_PATH if drive_mounted else None,
)

---
## ✂️ Section 3 — Subject-Level Split + Leakage Verification

> Training will be **aborted** if any subject appears in more than one split.

In [ ]:
# ── Split at the subject level ──────────────────────────────────────────────
from data.splitter import split_subjects, verify_no_leakage

split = split_subjects(
    records=records,
    train_ratio=cfg.data.train_ratio,
    val_ratio=cfg.data.val_ratio,
    test_ratio=cfg.data.test_ratio,
    seed=cfg.data.split_seed,
)

# ── LEAKAGE VERIFICATION — must pass before training ────────────────────────
verify_no_leakage(split)   # Raises RuntimeError if any overlap > 0

print(f"\n✅ Split summary:")
print(f"   Train : {len(split.train_subjects)} subjects ({len(split.train)} scans)")
print(f"   Val   : {len(split.val_subjects)} subjects ({len(split.val)} scans)")
print(f"   Test  : {len(split.test_subjects)} subjects ({len(split.test)} scans)")

---
## 🔬 Section 4 — Preprocessing

Preprocessing is **cached** locally (`/content/cache/`) to avoid re-reading from Google Drive during training.

| Step | MRI | PET |
|---|---|---|
| 1 | Load + validate affine | Load + validate pair |
| 2 | Reorient RAS+ | Co-register to MRI (rigid) |
| 3 | Resample 1mm³ | Tracer identification |
| 4 | Skull strip (HD-BET) | Tracer-specific normalization |
| 5 | N4 bias correction | Resample to MRI grid |
| 6 | Intensity normalization | Save to cache |
| 7 | Crop/pad to 128³ | — |
| 8 | Save to cache | — |

In [ ]:
# ── Check cache before preprocessing ───────────────────────────────────────
import shutil

CACHE_ROOT = Path(cfg.environment.cache_root)
PREPROC_CACHE = CACHE_ROOT / "preprocessed"

free_gb = shutil.disk_usage("/").free / 1024**3
n_subjects = len(records)
estimated_gb = n_subjects * 0.05   # rough: ~50 MB per subject pair

print(f"Local disk free : {free_gb:.1f} GB")
print(f"Est. cache size : ~{estimated_gb:.1f} GB ({n_subjects} subjects)")

existing = list(PREPROC_CACHE.rglob("mri.nii.gz")) if PREPROC_CACHE.exists() else []
print(f"\nAlready cached  : {len(existing)} MRI files")

SKIP_PREPROC = len(existing) >= n_subjects
if SKIP_PREPROC:
    print("\n✅ Full cache found — preprocessing will be skipped.")
else:
    print(f"\n⏳ {n_subjects - len(existing)} subjects need preprocessing.")

In [ ]:
# ── Run preprocessing ───────────────────────────────────────────────────────
from preprocessing.pipeline import run_preprocessing

# Convert all splits into one list for preprocessing
all_records = split.train + split.val + split.test
all_entries = [r.to_dict() for r in all_records]

preproc_results = run_preprocessing(
    subjects=all_entries,
    cfg=cfg,
    force=False,   # set True to reprocess even if cached
)

n_ok    = sum(r.success for r in preproc_results)
n_fail  = sum(not r.success for r in preproc_results)
print(f"\n✅ Preprocessing complete: {n_ok} succeeded, {n_fail} failed.")

if n_fail > 0:
    print("\n⚠️  Failed subjects:")
    for r in preproc_results:
        if not r.success:
            print(f"   {r.subject_id}/{r.visit_id}: {r.error}")

---
## 📦 Section 5 — Dataset & DataLoader

In [ ]:
# ── Build DataLoaders ───────────────────────────────────────────────────────
from data.dataset import build_dataloader

# Filter splits to successfully preprocessed subjects only
success_ids = {f"{r.subject_id}/{r.visit_id}" for r in preproc_results if r.success}

def filter_split(records_list):
    return [r for r in records_list if f"{r.subject_id}/{r.visit_id}" in success_ids]

train_records = filter_split(split.train)
val_records   = filter_split(split.val)
test_records  = filter_split(split.test)

train_loader = build_dataloader(train_records, CACHE_ROOT, "train", cfg)
val_loader   = build_dataloader(val_records,   CACHE_ROOT, "val",   cfg)
test_loader  = build_dataloader(test_records,  CACHE_ROOT, "test",  cfg)

print(f"✅ DataLoaders ready:")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val batches   : {len(val_loader)}")
print(f"   Test batches  : {len(test_loader)}")

In [ ]:
# ── Preview one batch ───────────────────────────────────────────────────────
import torch

batch = next(iter(train_loader))
print(f"MRI batch  : {batch['mri'].shape}  dtype={batch['mri'].dtype}")
print(f"PET batch  : {batch['pet'].shape}  dtype={batch['pet'].dtype}")
print(f"Subjects   : {batch['subject_id']}")
print(f"MRI range  : [{batch['mri'].min():.3f}, {batch['mri'].max():.3f}]")
print(f"PET range  : [{batch['pet'].min():.3f}, {batch['pet'].max():.3f}]")

---
## 🏗️ Section 6 — Model & Loss

In [ ]:
# ── Build model ─────────────────────────────────────────────────────────────
from models.factory import build_model

model = build_model(cfg.model)
model_cfg = model.get_config()

print(f"✅ Model built:")
print(f"   Architecture   : {model_cfg['name']}")
print(f"   Parameters     : {model_cfg['num_parameters']:,}")
print(f"   Features       : {model_cfg['features']}")
print(f"   Dropout        : {model_cfg['dropout']}")

# Quick shape test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
dummy = torch.randn(1, 1, *cfg.patch.size, device=device)
with torch.no_grad():
    out = model(dummy)
assert out.shape == dummy.shape, f"Shape mismatch: {out.shape} != {dummy.shape}"
print(f"   Shape test     : {dummy.shape} → {out.shape} ✅")
assert not torch.isnan(out).any(), "NaN in output!"
print(f"   NaN check      : ✅ clean")
del dummy, out

In [ ]:
# ── Build loss function ─────────────────────────────────────────────────────
from losses.combined import CombinedLoss

loss_fn = CombinedLoss(cfg.loss)

# Quick loss test
a = torch.randn(1, 1, *cfg.patch.size, device=device)
b = torch.randn(1, 1, *cfg.patch.size, device=device)
total, components = loss_fn(a, b)
print(f"✅ Loss test: total={total.item():.4f}, components={components}")
del a, b

---
## 🚀 Section 7 — Training

**Features:**
- Automatic Mixed Precision (AMP)
- Gradient accumulation (effective batch = `batch_size × gradient_accumulation_steps`)
- Automatic checkpoint resume
- Early stopping
- NaN detection → debug checkpoint saved automatically

In [ ]:
# ── Initialize experiment tracker ───────────────────────────────────────────
from tracking.tracker import ExperimentTracker

tracker = ExperimentTracker(
    run_dir=Path(cfg.environment.output_root) / "runs" / "current",
    config_dict=config_to_dict(cfg),
    wandb_enabled=cfg.tracking.wandb_enabled,
    wandb_project=cfg.tracking.wandb_project,
    experiments_index=Path(cfg.environment.output_root) / cfg.tracking.experiments_index,
)

RUN_DIR = Path(cfg.environment.output_root) / "runs" / tracker.run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Save run config
tracker.log_config(RUN_DIR / "run_config.yaml")

print(f"✅ Experiment tracker ready.")
print(f"   Run ID  : {tracker.run_id}")
print(f"   Run dir : {RUN_DIR}")

In [ ]:
# ── Check for existing checkpoint to resume from ────────────────────────────
from engine.colab import find_latest_checkpoint

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
resume_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR)

if resume_checkpoint:
    print(f"⏩ Resuming from: {resume_checkpoint.name}")
else:
    print("🆕 Starting training from scratch.")

In [ ]:
# ── Run training ────────────────────────────────────────────────────────────
# ⚠️  This cell may run for hours. It saves checkpoints to Drive automatically.
#     If the Colab session disconnects, re-run from this cell to auto-resume.

from training.trainer import Trainer

trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    run_dir=RUN_DIR,
)

try:
    trainer.train(resume_checkpoint=resume_checkpoint)
    print("\n🎉 Training complete!")
except RuntimeError as e:
    print(f"\n❌ Training failed: {e}")
    raise

---
## 📊 Section 8 — Evaluation on Test Set

In [ ]:
# ── Load best model ─────────────────────────────────────────────────────────
BEST_MODEL_PATH = RUN_DIR / "checkpoints" / "best_model.pt"

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model not found at {BEST_MODEL_PATH}. Run training first.")

model.load_state_dict(torch.load(str(BEST_MODEL_PATH), map_location=device))
model.eval()
print(f"✅ Best model loaded from: {BEST_MODEL_PATH.name}")

In [ ]:
# ── Run inference + compute metrics on test set ─────────────────────────────
import json
import numpy as np
import nibabel as nib
from inference.sliding_window import run_inference
from evaluation.metrics import compute_all_metrics, aggregate_metrics

INFERENCE_DIR = RUN_DIR / "inference"
INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

per_subject_metrics = []

for rec in test_records:
    mri_path = CACHE_ROOT / "preprocessed" / rec.subject_id / rec.visit_id / "mri.nii.gz"
    pet_path = CACHE_ROOT / "preprocessed" / rec.subject_id / rec.visit_id / "pet.nii.gz"

    if not mri_path.exists() or not pet_path.exists():
        print(f"  ⚠️  Skipping {rec.subject_id} — cache not found.")
        continue

    # Run inference
    inf_result = run_inference(
        model=model,
        mri_path=mri_path,
        output_dir=INFERENCE_DIR / rec.subject_id,
        cfg=cfg,
        subject_id=rec.subject_id,
        pet_path=pet_path,
    )

    # Compute metrics
    synth = nib.load(inf_result["synth_pet_path"]).get_fdata(dtype=np.float32)
    real  = nib.load(str(pet_path)).get_fdata(dtype=np.float32)
    mri   = nib.load(str(mri_path)).get_fdata(dtype=np.float32)
    brain_mask = (mri != 0).astype(np.uint8)

    metrics = compute_all_metrics(synth, real, brain_mask)
    metrics["subject_id"] = rec.subject_id
    metrics["inference_time_s"] = inf_result["inference_time_s"]
    per_subject_metrics.append(metrics)

    # Save per-subject metrics
    metric_path = INFERENCE_DIR / rec.subject_id / f"{rec.subject_id}_metrics.json"
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    with open(metric_path, "w") as f:
        json.dump(metrics, f, indent=2)

print(f"\n✅ Inference complete for {len(per_subject_metrics)} subjects.")

In [ ]:
# ── Print aggregate metrics ─────────────────────────────────────────────────
import pandas as pd

agg = aggregate_metrics(per_subject_metrics)

print("\n" + "═" * 60)
print("Test Set Evaluation Results")
print("═" * 60)
print(f"{'Metric':<12} {'Mean':>10} {'Std':>10} {'N':>6}")
print("─" * 40)
for k, v in agg.items():
    if k not in ["subject_id", "inference_time_s"]:
        print(f"{k.upper():<12} {v['mean']:>10.4f} {v['std']:>10.4f} {v['n']:>6}")
print("═" * 60)

# Save to CSV
df = pd.DataFrame(per_subject_metrics)
csv_path = RUN_DIR / "test_metrics.csv"
df.to_csv(str(csv_path), index=False)
print(f"\n✅ Per-subject metrics saved to: {csv_path.name}")

---
## 📈 Section 9 — Visualization

In [ ]:
# ── Training curves ─────────────────────────────────────────────────────────
from visualization.plots import plot_training_curves
import matplotlib.pyplot as plt
from IPython.display import Image, display

METRICS_JSONL = RUN_DIR / "logs" / "metrics.jsonl"
VIZ_DIR = RUN_DIR / "visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

if METRICS_JSONL.exists():
    plot_training_curves(METRICS_JSONL, VIZ_DIR)
    print("Training curves:")
    for png in VIZ_DIR.glob("*.png"):
        print(f"  {png.name}")
        display(Image(str(png)))
else:
    print("⚠️  metrics.jsonl not found — run training first.")

In [ ]:
# ── Axial overlays for test subjects ───────────────────────────────────────
from visualization.plots import plot_overlay
from IPython.display import Image, display

MAX_SUBJECTS_TO_PLOT = 3

for rec in test_records[:MAX_SUBJECTS_TO_PLOT]:
    mri_path  = CACHE_ROOT / "preprocessed" / rec.subject_id / rec.visit_id / "mri.nii.gz"
    pet_path  = CACHE_ROOT / "preprocessed" / rec.subject_id / rec.visit_id / "pet.nii.gz"
    synth_path = INFERENCE_DIR / rec.subject_id / f"{rec.subject_id}_synth_pet.nii.gz"

    if not synth_path.exists():
        print(f"  ⚠️  Synthesized PET not found for {rec.subject_id} — skipping.")
        continue

    mri   = nib.load(str(mri_path)).get_fdata(dtype=np.float32)
    real  = nib.load(str(pet_path)).get_fdata(dtype=np.float32)
    synth = nib.load(str(synth_path)).get_fdata(dtype=np.float32)

    overlay_path = VIZ_DIR / f"{rec.subject_id}_overlay.png"
    plot_overlay(mri, real, synth, overlay_path, subject_id=rec.subject_id)
    display(Image(str(overlay_path)))

---
## 🔬 Section 10 — Statistical Comparison (Multi-Experiment)

Use this section when comparing two or more experiments (e.g., different models, loss configs).

In [ ]:
# ── Statistical comparison ──────────────────────────────────────────────────
# This example compares the current experiment against itself (placeholder).
# To compare two real experiments:
#   1. Load per_subject_metrics from each experiment's test_metrics.csv
#   2. Pass both as entries in the `experiments` dict.

from evaluation.statistics import compare_experiments

# Placeholder: single experiment — no comparison needed
if len(per_subject_metrics) >= 5:
    # Example: compare first half vs second half (for demo only)
    half = len(per_subject_metrics) // 2
    exp_dict = {
        "experiment_A": per_subject_metrics[:half],
        "experiment_B": per_subject_metrics[half:],
    }
    results = compare_experiments(
        experiments=exp_dict,
        reference="experiment_A",
        metrics=["ssim", "psnr", "mae"],
        test=cfg.evaluation.statistical_test,
        correction=cfg.evaluation.multiple_comparison_correction,
    )
else:
    print("ℹ️  Need at least 5 subjects for meaningful statistical comparison.")

---
## 💾 Section 11 — Save Outputs to Google Drive

In [ ]:
# ── Copy final outputs to Drive ─────────────────────────────────────────────
import shutil

DRIVE_OUTPUT = Path(cfg.environment.output_root) / tracker.run_id
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

items_to_copy = [
    (RUN_DIR / "run_config.yaml",    DRIVE_OUTPUT / "run_config.yaml"),
    (RUN_DIR / "test_metrics.csv",   DRIVE_OUTPUT / "test_metrics.csv"),
    (RUN_DIR / "checkpoints" / "best_model.pt", DRIVE_OUTPUT / "best_model.pt"),
    (RUN_DIR / "logs" / "metrics.jsonl",         DRIVE_OUTPUT / "metrics.jsonl"),
]

for src, dst in items_to_copy:
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f"  ✅ Copied: {src.name} → Drive")
    else:
        print(f"  ⚠️  Not found: {src.name}")

# Copy visualizations
VIZ_DRIVE = DRIVE_OUTPUT / "visualizations"
if VIZ_DIR.exists():
    shutil.copytree(str(VIZ_DIR), str(VIZ_DRIVE), dirs_exist_ok=True)
    print(f"  ✅ Visualizations copied to Drive.")

# Update experiments index
tracker.log_final_metrics({
    **{k: v["mean"] for k, v in agg.items() if k not in ["subject_id"]},
    "model": cfg.model.name,
    "dataset": cfg.data.dataset,
    "n_train": len(train_records),
    "n_test": len(test_records),
})

tracker.finish()
print(f"\n🎉 All outputs saved to: {DRIVE_OUTPUT}")

---
## 🔄 Section 12 — Standalone Inference (New Subjects)

Use this section to run inference on new subjects **after** training.

In [ ]:
# ── Inference on a new MRI ──────────────────────────────────────────────────
# Adjust these paths for your new subject:

NEW_MRI_PATH     = Path("/content/drive/MyDrive/Google Colabs/data/new_subject/mri.nii.gz")
NEW_SUBJECT_ID   = "new_subject"
NEW_CHECKPOINT   = DRIVE_OUTPUT / "best_model.pt"   # or any other checkpoint

if NEW_MRI_PATH.exists():
    # Load model
    model.load_state_dict(torch.load(str(NEW_CHECKPOINT), map_location=device))
    model.eval()

    NEW_OUTPUT_DIR = DRIVE_OUTPUT / "new_inference" / NEW_SUBJECT_ID

    result = run_inference(
        model=model,
        mri_path=NEW_MRI_PATH,
        output_dir=NEW_OUTPUT_DIR,
        cfg=cfg,
        subject_id=NEW_SUBJECT_ID,
        pet_path=None,   # no ground truth available
    )
    print(f"✅ Synthesized PET saved to: {result['synth_pet_path']}")
    print(f"   Inference time : {result['inference_time_s']:.1f}s")
else:
    print(f"ℹ️  Set NEW_MRI_PATH to a real MRI file to use standalone inference.")

---

## ✅ Pipeline Complete

| Step | Status |
|---|---|
| Environment setup | ✅ |
| Dataset inspection | ✅ |
| Subject-level split + leakage verification | ✅ |
| MRI + PET preprocessing (cached) | ✅ |
| Model training with AMP + checkpoint | ✅ |
| Evaluation (SSIM/PSNR/MAE/MSE/NMSE/PCC) | ✅ |
| Visualization (curves + overlays) | ✅ |
| Outputs saved to Google Drive | ✅ |

**Next steps:**
- Compare this run against alternative models via Section 10
- Add clinical SUVr evaluation for Amyloid/Tau tracers
- Try `config.model.name = 'swin_unetr'` for a transformer-based baseline

---
*Generated by the AI Research Team Pipeline — v3*